# Notebook 05 — Query Engine

**Prerequisites:**
1. Notebook 04 completed (docs ingested into ChromaDB)
2. Postgres running + migrated
3. Ollama running with models loaded

**What to validate:**
- End-to-end query (Executive Summary mode)
- Detailed mode with citations
- No-access case (CrossDomainPermissionRequired signal)
- Domain filter restricts cross-domain
- Conversation persistence verification

In [ ]:
import sys
sys.path.insert(0, '..')

import uuid
from sqlalchemy import select

from policy_system.db.session import get_session
from policy_system.db.models import User, ResponseFormat
from policy_system.llm.ollama_provider import OllamaProvider
from policy_system.rag.chromadb_provider import ChromaDBProvider
from policy_system.query.engine import run_query
from policy_system.query.conversation_service import get_conversation_messages
from policy_system.core.models import CrossDomainPermissionRequired

llm = OllamaProvider()
# Use the same collection from notebook 04
rag = ChromaDBProvider(persist_dir='../data/chroma_db', collection_name='notebook_04_test')

print(f'ChromaDB chunks: {rag.get_chunk_count()}')
print('Providers ready')

## 1. End-to-End Query — Executive Summary Mode

In [ ]:
async def test_summary_query():
    async with get_session() as session:
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
        
        result = await run_query(
            session=session,
            rag_provider=rag,
            llm_provider=llm,
            user=it_user,
            message='What are the password requirements?',
            format_override=ResponseFormat.EXECUTIVE_SUMMARY,
        )
        
        print(f'Response type: {type(result).__name__}')
        print(f'Format used: {result.format_used}')
        print(f'Docs retrieved: {result.retrieved_doc_ids}')
        print(f'\nResponse:\n{result.content}')
        return result

summary_result = await test_summary_query()

## 2. Detailed Mode with Citations

In [ ]:
from policy_system.query.citation_builder import build_citations

async def test_detailed_query():
    async with get_session() as session:
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
        
        result = await run_query(
            session=session,
            rag_provider=rag,
            llm_provider=llm,
            user=it_user,
            message='Explain the data classification levels and what Restricted data requires.',
            format_override=ResponseFormat.DETAILED_RESPONSE,
        )
        
        citations = build_citations(result.retrieved_chunks)
        
        print(f'Format: {result.format_used}')
        print(f'Citations found: {len(citations)}')
        for c in citations:
            print(f'  [{c.doc_title}, Page {c.page_number}, Para {c.para_number}]')
        print(f'\nResponse (first 500 chars):\n{result.content[:500]}')
        return result

detailed_result = await test_detailed_query()

## 3. No-Access Case — CrossDomainPermissionRequired Signal

In [ ]:
async def test_no_access():
    async with get_session() as session:
        # IT user asking about finance — should get CrossDomainPermissionRequired
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
        
        result = await run_query(
            session=session,
            rag_provider=rag,
            llm_provider=llm,
            user=it_user,
            message='What are the expense reporting requirements?',
            # IT user's role doesn't cover finance docs, so 0 chunks will match
        )
        
        print(f'Result type: {type(result).__name__}')
        
        if isinstance(result, CrossDomainPermissionRequired):
            print(f'Message: {result.message}')
            print(f'Available domains: {result.available_domains}')
            print('LLM was NOT called (correct behavior)')
            print('CrossDomainPermissionRequired test: PASS')
        else:
            print('WARNING: Expected CrossDomainPermissionRequired but got a response.')
            print('This may mean the IT user has access to finance docs via ChromaDB test data.')

await test_no_access()

## 4. Conversation Persistence

In [ ]:
async def test_conversation_persistence():
    async with get_session() as session:
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
        
        # First query — new conversation
        result1 = await run_query(
            session=session,
            rag_provider=rag,
            llm_provider=llm,
            user=it_user,
            message='What is the MFA requirement?',
        )
        
        if isinstance(result1, CrossDomainPermissionRequired):
            print('No IT docs in ChromaDB yet — run notebook 04 first')
            return
        
        conv_id = result1.conv_id
        print(f'Conversation created: {conv_id}')
        
        # Follow-up query — same conversation
        result2 = await run_query(
            session=session,
            rag_provider=rag,
            llm_provider=llm,
            user=it_user,
            message='What happens if an incident occurs?',
            conv_id=conv_id,
        )
        
        assert result2.conv_id == conv_id, 'Should continue same conversation'
        
        # Verify messages are in DB
        messages = await get_conversation_messages(session, conv_id)
        print(f'Messages in conversation: {len(messages)}')
        for msg in messages:
            print(f'  [{msg.role}] {msg.content[:80]}...')
        
        assert len(messages) == 4, f'Expected 4 messages (2 user + 2 assistant), got {len(messages)}'
        print('Conversation persistence: PASS')

await test_conversation_persistence()

## Summary

Query engine validation complete:
- Executive Summary mode ✓
- Detailed mode with citations ✓
- CrossDomainPermissionRequired signal (LLM not called) ✓
- Conversation persistence ✓

**Next:** Notebook 06 — Feedback & Flagging